In [ ]:
 %run /Users/manojrammopati/Downloads/bupa_insurance_project/00_Pre_Pilot/00_Pre_Pilot/_00__Pre_Connectors.ipynb

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts


25/11/25 05:08:46 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1068991 ms exceeds timeout 120000 ms
25/11/25 05:08:46 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/25 05:08:54 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at 

In [ ]:
# Databricks: install Kaggle
%pip install -q kaggle
dbutils.library.restartPython()

In [ ]:
# (A) Put your Kaggle API token JSON content here if you have it securely:
# kaggle.json content looks like: {"username":"<user>","key":"<key>"}
KAGGLE_JSON = """
{"username":"manojrammopati1","key":"14c734c72426deb6838c1b4b15f860f2"}
""".strip()

# Save to the expected path in the driver
import os, json
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json","w") as f:
    f.write(KAGGLE_JSON)
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle token written.")

In [ ]:
# Create a temp download folder
download_dir = "/dbfs/tmp/kaggle_bupa"
import os, shutil, subprocess, textwrap

if os.path.exists(download_dir):
    shutil.rmtree(download_dir)
os.makedirs(download_dir, exist_ok=True)

def kaggle_download(slug, target):
    # Uses CLI to download + unzip
    cmd = f"kaggle datasets download -d {slug} -p {target} --unzip"
    print("Running:", cmd)
    out = subprocess.run(cmd.split(), capture_output=True, text=True)
    print(out.stdout or out.stderr)

kaggle_download("rohitrox/healthcare-provider-fraud-detection-analysis", download_dir)
kaggle_download("anmolkumar/health-insurance-cross-sell-prediction", download_dir)

# List what we got:
import glob
files = glob.glob(download_dir+"/*")
files

In [ ]:
bronze_container = "rawdatakaggle"  # recommended
bronze_base = f"abfss://{bronze_container}@{storage_account1}.dfs.core.windows.net"

paths = {
    "policies":   f"{bronze_base}/policies",
    "members":    f"{bronze_base}/members",
    "claims":     f"{bronze_base}/claims",
    "providers":  f"{bronze_base}/providers"
}
paths

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Try common filenames (some forks differ slightly)
base = "/tmp/kaggle_bupa"  # driver path (without /dbfs) works with spark.read.csv
import os

def try_read(names):
    for n in names:
        p = os.path.join(base, n)
        if os.path.exists("/dbfs"+p):  # DBFS path exists?
            return spark.read.option("header", True).csv(p)
    return None

# Provider fraud dataset parts
inpatient   = try_read(["Inpatientdata.csv","inpatientdata.csv"])
outpatient  = try_read(["Outpatientdata.csv","outpatientdata.csv"])
beneficiary = try_read(["Beneficiarydata.csv","beneficiarydata.csv"])
potential   = try_read(["PotentialFraud.csv","potentialfraud.csv","Train-1542865627584.csv","Provider.csv"])

# Cross-sell dataset
crosssell  = try_read(["train.csv","Train.csv","health_insurance_train.csv"])

display(inpatient.limit(5) if inpatient else None)
display(outpatient.limit(5) if outpatient else None)
display(beneficiary.limit(5) if beneficiary else None)
display(potential.limit(5) if potential else None)
display(crosssell.limit(5)  if crosssell  else None)

In [ ]:
# Many versions label fraud at provider-level in 'PotentialFraud.csv' mapping PROVIDER_ID -> potential fraud (Y/N)
# Normalize to Provider_ID + Fraud_Flag

prov_id_col = "Provider"  # typical column name
if potential and prov_id_col not in potential.columns:
    # try other common names
    for c in potential.columns:
        if c.lower() in ("provider","provider_id","providerid"):
            prov_id_col = c
            break

providers_raw = potential.select(
    F.col(prov_id_col).alias("Provider_ID"),
    *[F.col(c) for c in potential.columns if c.lower() != prov_id_col.lower()]
)

# Create Fraud_Flag if not present: if a column named 'PotentialFraud' exists, use it; else default 0
fraud_col = None
for c in providers_raw.columns:
    if c.lower() in ("potentialfraud","fraud_flag","fraudlabel","is_fraud"):
        fraud_col = c
        break

if fraud_col:
    providers_bronze = providers_raw.select(
    F.col("Provider_ID"),
    F.when(
        F.lower(F.col(fraud_col)).isin("yes", "y", "true", "1"), 1
    ).when(
        F.expr(f"try_cast({fraud_col} as int)") == 1, 1
    ).otherwise(0).alias("Fraud_Flag")
).dropDuplicates(["Provider_ID"])
else:
    providers_bronze = providers_raw.select("Provider_ID").dropDuplicates().withColumn("Fraud_Flag", F.lit(0))

display(providers_bronze.limit(10))

In [ ]:
import glob

def try_read(patterns):
    for pat in patterns:
        # Use glob to find matching files
        matches = glob.glob("/dbfs/tmp/kaggle_bupa/" + pat)
        if matches:
            # Remove /dbfs prefix for Spark
            spark_path = matches[0].replace("/dbfs", "")
            return spark.read.option("header", True).csv(spark_path)
    return None

# Update patterns to use wildcards
inpatient   = try_read(["*Inpatientdata*.csv"])
outpatient  = try_read(["*Outpatientdata*.csv"])
beneficiary = try_read(["*Beneficiarydata*.csv"])
potential   = try_read(["*PotentialFraud*.csv", "*Provider*.csv", "*Train-*.csv"])

crosssell   = try_read(["train.csv", "Train.csv", "health_insurance_train.csv"])

display(inpatient.limit(5) if inpatient else None)
display(outpatient.limit(5) if outpatient else None)
display(beneficiary.limit(5) if beneficiary else None)
display(potential.limit(5) if potential else None)
display(crosssell.limit(5)  if crosssell  else None)

In [ ]:
# These files usually have: 'ClaimID','Provider','BeneID','ClaimStartDt','ClaimEndDt','InscClaimAmtReimbursed', etc.
# We'll unify to a Bupa-like schema and keep keys to join Providers later.

import pyspark.sql.functions as F

# Load the CSVs (adjust the file paths as needed)
# Do NOT reload CSVs here; use inpatient and outpatient from Cell 6

def unify_claims(df):
    if df is None: 
        return None
    c = {col.lower(): col for col in df.columns}
    return df.select(
        F.col(c.get("claimid","ClaimID")).alias("Claim_ID"),
        F.col(c.get("provider","Provider")).alias("Provider_ID"),
        F.col(c.get("beneid","BeneID")).alias("Member_Key"),
        F.try_to_date(F.col(c.get("claimstartdt","ClaimStartDt")), "MM/dd/yyyy").alias("Date_Reported"),
        F.try_to_date(F.col(c.get("claimenddt","ClaimEndDt")), "MM/dd/yyyy").alias("Date_Settled"),
        F.col(c.get("inscclaimamtreimbursed","InscClaimAmtReimbursed")).cast("double").alias("Payout_GBP")
    )

clm_in  = unify_claims(inpatient)
clm_out = unify_claims(outpatient)

# ... rest of your code ...

claims_union = None
if clm_in and clm_out:
    claims_union = clm_in.unionByName(clm_out, allowMissingColumns=True)
elif clm_in:
    claims_union = clm_in
elif clm_out:
    claims_union = clm_out
else:
    raise Exception("Could not find inpatient/outpatient claims CSVs; adjust filenames and re-run step 3.")

# Continue with your enrichment logic...

# Add synthetic columns to fit target schema
# (We’ll synthesize Claim_Amount_GBP, Fraud_Label, Claim_Type, Claim_Status and Policy_ID)
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Claim amount as payout + a random margin (can be < payout in some anomalies)
claims_enriched = (claims_union
  .withColumn("Claim_Amount_GBP",
              (F.col("Payout_GBP") * F.rand(seed=7) * 0.6 + F.col("Payout_GBP")).cast("double"))
  .withColumn("Fraud_Label", (F.rand(seed=9) < F.lit(0.015)).cast("int"))  # ~1.5%
  .withColumn("Claim_Type", F.when(F.rand(seed=11) < 0.45,"Hospital")
                            .when(F.rand(seed=12) < 0.65,"Outpatient")
                            .when(F.rand(seed=13) < 0.72,"Dental")
                            .when(F.rand(seed=14) < 0.77,"Maternity")
                            .when(F.rand(seed=15) < 0.80,"Travel")
                            .otherwise("Physio"))
  .withColumn("Claim_Status", F.when(F.rand(seed=21) < 0.70,"Settled")
                               .when(F.rand(seed=22) < 0.88,"Pending")
                               .when(F.rand(seed=23) < 0.98,"Rejected").otherwise("Withdrawn"))
)

# introduce ~7% nulls in Provider_ID and Claim_Type
claims_enriched = (claims_enriched
  .withColumn("Provider_ID", F.when(F.rand(seed=31) < 0.07, F.lit(None)).otherwise(F.col("Provider_ID")))
  .withColumn("Claim_Type", F.when(F.rand(seed=32) < 0.07, F.lit(None)).otherwise(F.col("Claim_Type")))
)

display(claims_enriched.limit(10))

In [ ]:
from pyspark.sql import functions as F

# Map original column names from the Kaggle cross-sell file
c = {col.lower(): col for col in crosssell.columns}
age_col       = c.get("age","Age")
prem_col      = c.get("annual_premium","Annual_Premium")
region_col    = c.get("region_code","Region_Code")
channel_col   = c.get("policy_sales_channel","Policy_Sales_Channel")
gender_col    = c.get("gender","Gender")
veh_damage_col= c.get("vehicle_damage","Vehicle_Damage")
id_col        = c.get("id","id")

# Helpers: safe numeric parsing using SQL try_cast (no ANSI errors)
def try_double(expr_str):
    # strip spaces/commas/letters, keep digits and one dot, then try_cast to double
    return F.expr(f"""
      try_cast(
        regexp_replace(
          regexp_replace({expr_str}, '[,\\s]', ''),    -- remove commas/whitespace
          '[^0-9\\.]', ''                              -- keep digits and dot
        )
      as double)
    """)

def try_int(expr_str):
    # remove non-digits (incl. dot), then try_cast to int
    return F.expr(f"""
      try_cast(
        regexp_replace({expr_str}, '[^0-9]', '')
      as int)
    """)

crosssell_clean = (
    crosssell
      # Age: tolerant parse -> floor to int -> bound to [0, 110]
      .withColumn("Age_double", try_double(f"`{age_col}`"))
      .withColumn("Age_int",
                  F.when(F.col("Age_double").isNull(), F.lit(None).cast("int"))
                   .otherwise(F.floor(F.col("Age_double")).cast("int")))
      .withColumn("Age_int",
                  F.when((F.col("Age_int") >= 0) & (F.col("Age_int") <= 110), F.col("Age_int")).otherwise(F.lit(None).cast("int")))
      # Annual_Premium: tolerant parse to double
      .withColumn("Annual_Premium_double", try_double(f"`{prem_col}`"))
      # Region as string code (keep numeric characters only)
      .withColumn("Region_code_str",
                  F.expr(f"regexp_replace(`{region_col}`, '[^0-9]', '')"))
      .withColumn("Region_code_str",
                  F.when(F.length(F.col("Region_code_str"))==0, None).otherwise(F.col("Region_code_str")))
      # Policy_Sales_Channel: tolerant int (this is where '151.0' was breaking)
      .withColumn("Policy_Sales_Channel_int", try_int(f"`{channel_col}`"))
)

# (Optional) sanity check
crosssell_clean.select("Age_int","Annual_Premium_double","Policy_Sales_Channel_int","Region_code_str").summary().show(truncate=False)

In [ ]:
# POLICIES
policy_base = (crosssell_clean
  .withColumn("Policy_ID",  F.concat(F.lit("P_"), F.lpad(F.col(id_col).cast("string"), 8, "0")))
  .withColumn("Customer_ID",F.concat(F.lit("CUST_"), F.lpad(F.col(id_col).cast("string"), 7, "0")))
  .withColumn("Annual_Premium_GBP", F.col("Annual_Premium_double"))
  .withColumn("Product_Line",
              F.when(F.rand(1) < 0.60,"Health")
               .when(F.rand(2) < 0.70,"Dental")
               .when(F.rand(3) < 0.85,"Motor")
               .when(F.rand(4) < 0.95,"Accident").otherwise("Travel"))
  .withColumn("Channel",
              F.when(F.col("Policy_Sales_Channel_int") < 20,"Online")
               .when(F.col("Policy_Sales_Channel_int") < 40,"Agent")
               .when(F.col("Policy_Sales_Channel_int") < 70,"Broker")
               .otherwise("Partner"))
  .withColumn("Policy_Start_Date", F.expr("date_sub(current_date(), cast(rand(5)*365*5 as int))"))
  .withColumn("Policy_End_Date",   F.expr("date_add(Policy_Start_Date, cast(180 + rand(6)*360 as int))"))
  .withColumn("Sum_Insured_GBP", (F.col("Annual_Premium_GBP")/F.lit(0.02) * (1+F.rand(7)*0.3)).cast("double"))
  .withColumn("Renewal_Offered_Flag", (F.rand(8) < 0.75).cast("int"))
  .withColumn("Renewal_Accepted_Flag", (F.rand(9) < 0.70).cast("int"))
  .withColumn("Discount_Offered_%", F.when(F.col("Renewal_Offered_Flag")==1, (F.rand(10)*20).cast("double")).otherwise(F.lit(0.0)))
  .select("Policy_ID","Customer_ID","Product_Line","Sum_Insured_GBP","Annual_Premium_GBP",
          "Policy_Start_Date","Policy_End_Date","Renewal_Offered_Flag","Renewal_Accepted_Flag",
          "Discount_Offered_%","Channel")
)

# MEMBERS
members_base = (crosssell_clean
  .withColumn("Member_ID", F.concat(F.lit("MEM_"), F.lpad(F.col(id_col).cast("string"), 8, "0")))
  .withColumn("Policy_ID", F.concat(F.lit("P_"), F.lpad(F.col(id_col).cast("string"), 8, "0")))
  .withColumn("Age", F.col("Age_int"))
  .withColumn("Gender", F.col(gender_col))
  .withColumn("BMI", (F.lit(27.5) + (F.rand(1)-0.5)*12).cast("double"))
  .withColumn("Smoker", F.when(F.col(veh_damage_col)=="Yes","Y").otherwise("N"))
  .withColumn("Chronic_Disease", F.when(F.rand(2)<0.12,"Diabetes")
                                  .when(F.rand(3)<0.27,"Hypertension")
                                  .when(F.rand(4)<0.35,"Asthma")
                                  .otherwise("None"))
  .withColumn("Employment_Status", F.when(F.rand(5)<0.55,"Employed")
                                     .when(F.rand(6)<0.67,"Retired")
                                     .when(F.rand(7)<0.75,"Student")
                                     .otherwise("Self-Employed"))
  .withColumn("Region", F.col("Region_code_str"))
  .select("Member_ID","Policy_ID","Age","Gender","BMI","Smoker","Chronic_Disease","Employment_Status","Region")
)

# quick check
policy_base.printSchema()
members_base.printSchema()
display(policy_base.limit(5))
display(members_base.limit(5))

In [ ]:
# Sample 1:1 mapping by hashing Claim_ID to a Policy_ID from cross-sell
from pyspark.sql.functions import col, sha2, concat_ws

policy_ids_df = policy_base.select("Policy_ID").withColumn("hash_key", F.rand(seed=42))
policy_ids_df = policy_ids_df.orderBy("hash_key").withColumn("rownum", F.monotonically_increasing_id())

claims_keyed = (claims_enriched
                .withColumn("rownum", F.monotonically_increasing_id()))

# Join by rownum modulo approach (approx uniform assignment)
pol_count = policy_ids_df.count()
claims_keyed = claims_keyed.withColumn("join_key", (F.col("rownum") % F.lit(pol_count)))

policy_ids_df = policy_ids_df.withColumn("join_key", (F.col("rownum") % F.lit(pol_count)))\
                             .select("Policy_ID","join_key")

claims_final = (claims_keyed.join(policy_ids_df, "join_key", "left")
                             .drop("rownum","hash_key","join_key"))

display(claims_final.limit(5))